[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab09_working_with_ai.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 9 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 9 — How analysts work with AI now
*Week 9. Baseline: sections 1–5. Powerup: section 6.*

In week 3 you wrote data-quality checks by hand. This week you do the same kind of task with an AI
assistant (Gemini, built into Colab) doing the typing, and you keep a log of every correction you had to make.
The question is not whether the AI is good. It is: **how do you know** when it is right?

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

**This week the AI is the point.** Use it for every step of sections 3 and 4. Your grade is the quality of
your verification and your log, not how much code you wrote.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


## 1. The rules for working with an assistant
**Assistance, verification, ownership.** The assistant writes; you verify; you own the result. In practice:

1. **Read every line it writes before you run it.** If you cannot explain a line, ask it to explain, then check
   the explanation against the documentation or the data.
2. **Verify with something it did not write.** A count you compute yourself, a value you look up in the raw
   rows, a second method. An AI checking its own work is not a check.
3. **Log every correction**, small ones included (section 5). The log is how you learn what to watch for.

**What AI gets wrong in data work** (watch for these today):
- It assumes clean types: re-reads files without `dtype=str`, so IDs and ZIP codes lose leading zeros.
- It turns `Not Available` into `NaN` and then into nothing, or into 0, without telling you.
- It uses textbook rules where the data has its own: a 50-state list, a "6 digits" ID pattern.
- It invents column names, or uses one that is close (`Facility_ID`, `Overall Rating`).
- It answers confidently from empty data. A table of `NaN`s with a heading is not an answer.
- It says "all checks pass" without having run anything.

**Using Colab's Gemini.** In an empty code cell, click **generate** (in the cell's "start coding or generate with AI" prompt), or open the **Gemini** chat panel (the spark icon at the top right or bottom of the window; the exact button moves between Colab releases), and describe what you want. Tell it the
DataFrame names (`hospitals`, `uhv`) and that every column is text. Keep each prompt in your log.
One Colab habit trips people up: Colab never re-runs cells for you. When the AI changes `check`, run that cell
again and every cell below it (**Runtime → Run cell and below**), or you will be looking at the old answer.

**No Gemini on your account?** Some university Google accounts have it switched off. Use any assistant in
another browser tab (claude.ai, for example), paste its code into the cell, and log it the same way.

## 2. The data
Two live CMS files, read as text exactly as in week 3: *Hospital General Information* (`hospitals`, one row per
hospital) and *Unplanned Hospital Visits* (`uhv`, one row per hospital × measure). Run the cell; nothing to report.

In [ ]:
hospitals = load_cms("hospital_general_info")
uhv = load_cms("unplanned_visits")
print("hospitals:", hospitals.shape, "| uhv:", uhv.shape)
print("measures in uhv:", sorted(uhv["Measure ID"].unique()))
hospitals.head(3)


## 3. Task A: the week 3 checks, with an assistant
**First, without AI (5 minutes).** Write the ZIP code check yourself in the cell below: how many rows of
`hospitals` have a `ZIP Code` that is not exactly 5 digits? This keeps your own skill warm and gives you one
number the AI cannot have influenced.

**Then, with AI.** Ask Colab's Gemini to write a function `check(df)` that returns one row per check
(`check`, `failures`, `example`) for these seven rules, and run it on `hospitals`:
1. `Facility ID` is unique
2. `Facility ID` is exactly 6 characters
3. `State` is a two-letter uppercase code
4. `ZIP Code` is 5 digits
5. `Hospital overall rating` is one of 1–5 or `Not Available`
6. `Emergency Services` is exactly `Yes` or `No`
7. no column has leading or trailing whitespace

**Baseline:** for every check that reports failures, look at the failing rows and decide: **real defect, or a
wrong check?** For every check that reports zero, prove it could catch a failure (break one value in a copy and
rerun). Log each correction in section 5.

In [ ]:
# TODO (no AI): how many ZIP codes are not exactly 5 digits?


In [ ]:
# TODO: generate check(df) with Colab's Gemini here (or in a new cell just below), then run check(hospitals).


In [ ]:
# TODO: your verification. For each check with failures, look at the failing rows.
# For each check with zero failures, break a value on purpose and show the check catches it, e.g.
# broken = hospitals.copy(); broken.loc[0, "State"] = "ct"; check(broken)


## 4. Task B: a new file, and a trap
Now the assistant works on a file you have not checked before. Ask it, one prompt at a time:

1. "Check that `uhv` has one row per `Facility ID` and `Measure ID`, that every `Score` is a number or
   `Not Available`, and that every `Facility ID` appears in `hospitals`."
2. "For each `Measure ID` in `uhv`, how many hospitals have a numeric `Score`?"
3. "Using `uhv`, which five states have the highest average hospital-wide readmission rate (`Measure ID`
   `Hybrid_HWR`)?"

**Baseline:** record what it produced for each prompt. Verify prompts 1 and 2 with your own code. For prompt 3,
decide whether the answer means anything, and prove it with one line of your own.

In [ ]:
# TODO: prompts 1 and 2 with Colab's Gemini, here or in new cells below.


In [ ]:
# TODO: prompt 3 with Colab's Gemini. Then, below its answer, your one-line check.


## 5. The AI correction log
**Baseline:** one row per correction, in order. A "correction" is anything you changed, rejected, or had to
re-prompt for: wrong logic, wrong column, a silent assumption, a missing case, an explanation that was wrong,
an answer that should not exist. If the AI was right the first time, log the prompt anyway with "none" in the
middle columns; a run of "none"s is a finding too.

| # | Prompt (short) | What the AI produced | What was wrong | How you verified | Your fix |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |
| 5 | | | | | |
| 6 | | | | | |

**Totals:** prompts *n* · corrections *n* · corrections you would have missed without week 3 *n*

**Keeping your own skills (three sentences):** which correction depended on something you learned by doing
week 3 by hand? Which part of today could you now do without the assistant? What will you check first the
next time an AI hands you a data-quality report?

## 6. Powerup: an assistant outside the notebook
Pick one. Both run on your own laptop, not in Colab. The course repository is private: ask the instructor for
read access, or use your own capstone repository instead.

**(a) Claude Code against the course repo.** Install Claude Code (instructions at https://code.claude.com/docs;
it needs a Claude subscription or an Anthropic API key). Clone the repository, `cd` into it, run `claude`, and
ask it three things: *"Explain what `load_cms` in labs/course_data.py does, line by line"*, *"What would break if
`load_cms` dropped `dtype=str`? Show me with a test"*, and *"Find every place a CMS 'Not Available' value could
become a number"*. Verify one claim from each answer yourself and add the results to your log.

**(b) An Anthropic SDK script.** Install the SDK (`pip install anthropic`), set `ANTHROPIC_API_KEY` in your shell
(never in the file), and run a script like the one below, which sends a course file to Claude for review.
Then check one of its claims by hand.

```python
# review.py  -  usage: python review.py labs/course_data.py
import sys, anthropic

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
code = open(sys.argv[1], encoding="utf-8").read()
response = client.beta.messages.create(
    model="claude-opus-5-5",
    max_tokens=16000,
    output_config={"effort": "medium"},
    betas=["server-side-fallback-2026-07-01"], fallbacks="default",
    messages=[{"role": "user", "content":
        "Review this data-loading code for silent data-quality risks (types, missing values, "
        "leading zeros, network failures). List each risk with the line it is on.\n\n" + code}],
)
if response.stop_reason == "refusal":
    print("The model declined the request.")
for block in response.content:
    if block.type == "text":
        print(block.text)
print(f"tokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")
```

**What to submit:** the three prompts or the script output, and one claim you verified (or disproved).

**Powerup notes:** *(edit this cell)*